# **Experimento: Modelos Avanzados â€” Spaceship Titanic**

**Objetivo:** Superar el accuracy del modelo base (NB04) utilizando modelos mÃ¡s sofisticados.

| Notebook | Rol en esta cadena |
|---|---|
| NB03 | Dataset procesado: 8,514 filas Ã— 35 features |
| NB04 | Baseline con LR, RF, GradientBoosting + GridSearchCV |
| **NB04.1 (este)** | Experimento con modelos avanzados buscando superar el target |

**Target de Ã©xito:** `TARGET_ACCURACY = 0.83`  
**Dataset:** `data/processed/train_features_scaled.csv` (producido por NB03)


## **1. ConfiguraciÃ³n y LibrerÃ­as**


In [19]:
# ── Parámetro de éxito del experimento
TARGET_ACCURACY = 0.83

# ── Rutas
DATA_PATH       = '../../data/processed/train_scaled.csv'
NB04_META_PATH  = '../../models/model_metadata.json'
MODEL_OUT_PATH  = '../../models/best_advanced_model.pkl'
META_OUT_PATH   = '../../models/advanced_model_metadata.json'


In [20]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import json
import warnings

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, RandomizedSearchCV,
)
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    ExtraTreesClassifier,
    StackingClassifier,
    VotingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


## **2. Cargar Dataset y Resultados de NB04**


In [21]:
df = pd.read_csv(DATA_PATH)
target = 'Transported'
X = df.drop(columns=[target])
y = df[target]

print(f'Dataset: {X.shape[0]:,} filas x {X.shape[1]} features')
print(f'Target balance: {y.mean():.1%} True | {1-y.mean():.1%} False')


Dataset: 8,514 filas x 35 features
Target balance: 50.4% True | 49.6% False


In [22]:
with open(NB04_META_PATH) as f:
    nb04_meta = json.load(f)

nb04_val_acc = nb04_meta['val_accuracy']
print('Resultados de NB04 (referencia):')
for k, v in nb04_meta.items():
    print(f'  {k}: {v}')


Resultados de NB04 (referencia):
  model: Gradient Boosting
  params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
  cv_accuracy: 0.8121
  val_accuracy: 0.808
  val_roc_auc: 0.8974
  n_features: 35
  n_train_samples: 6811


## **3. Split Train / Validation**


In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}')


Train: 6,811  Val: 1,703


## **4. Modelos Avanzados â€” EvaluaciÃ³n con CV**

- **HistGradientBoostingClassifier:** equivalente a XGBoost/LightGBM dentro de sklearn, muy eficiente
- **ExtraTreesClassifier:** variante de Random Forest con mayor aleatorizaciÃ³n, rÃ¡pido
- **XGBoost / LightGBM:** si estÃ¡n disponibles en el entorno


In [24]:
advanced_models = {
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=200, random_state=42
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, random_state=42,
        eval_metric='logloss', verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, random_state=42, verbose=-1
    ),
}

adv_results = {}
for name, model in advanced_models.items():
    cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    adv_results[name] = {
        'cv_acc': cv_acc.mean(),
        'cv_acc_std': cv_acc.std(),
        'cv_auc': cv_auc.mean(),
    }
    print(f'{name:25s}  acc={cv_acc.mean():.4f} +/-{cv_acc.std():.4f}  auc={cv_auc.mean():.4f}')


HistGradientBoosting       acc=0.8105 +/-0.0085  auc=0.9010
ExtraTrees                 acc=0.7865 +/-0.0058  auc=0.8653
XGBoost                    acc=0.8011 +/-0.0061  auc=0.8941
LightGBM                   acc=0.8063 +/-0.0073  auc=0.8990


In [25]:
adv_results_df = pd.DataFrame(adv_results).T.sort_values('cv_acc', ascending=False)

fig = px.bar(
    adv_results_df.reset_index().rename(columns={'index': 'Modelo'}),
    x='Modelo', y='cv_acc', error_y='cv_acc_std',
    title=f'CV Accuracy â€” Modelos Avanzados (target: {TARGET_ACCURACY})',
    labels={'cv_acc': 'CV Accuracy'},
    color='cv_acc', color_continuous_scale='Greens',
)
if nb04_val_acc:
    fig.add_hline(y=nb04_val_acc, line_dash='dash', line_color='red',
                  annotation_text=f'NB04 referencia: {nb04_val_acc:.3f}')
fig.add_hline(y=TARGET_ACCURACY, line_dash='dot', line_color='orange',
              annotation_text=f'Target: {TARGET_ACCURACY}')
fig.update_layout(height=400, showlegend=False)
fig.show()


## **5. Tuning del Mejor Modelo Avanzado**

RandomizedSearchCV permite explorar un espacio de hiperparÃ¡metros mÃ¡s amplio que GridSearch.


In [26]:
best_adv_name = adv_results_df.index[0]
print(f'Modelo para tuning: {best_adv_name}')

param_spaces = {
    'HistGradientBoosting': (
        HistGradientBoostingClassifier(random_state=42),
        {
            'max_iter': [100, 200, 300, 500],
            'max_leaf_nodes': [15, 31, 63, 127],
            'max_depth': [None, 5, 10, 20],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'l2_regularization': [0.0, 0.1, 1.0],
            'min_samples_leaf': [10, 20, 50],
        },
    ),
    'ExtraTrees': (
        ExtraTreesClassifier(random_state=42, n_jobs=-1),
        {
            'n_estimators': [200, 300, 500],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
        },
    ),
    'XGBoost': (
        XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
        {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7, 9],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'subsample': [0.6, 0.8, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
            'reg_alpha': [0, 0.1, 1.0],
            'reg_lambda': [1.0, 1.5, 2.0],
        },
    ),
    'LightGBM': (
        LGBMClassifier(random_state=42, verbose=-1),
        {
            'n_estimators': [100, 200, 300],
            'max_depth': [-1, 5, 10, 15],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'num_leaves': [31, 63, 127],
            'subsample': [0.6, 0.8, 1.0],
            'colsample_bytree': [0.6, 0.8, 1.0],
        },
    ),
}

base_est, param_dist = param_spaces[best_adv_name]

rand_search = RandomizedSearchCV(
    base_est, param_dist,
    n_iter=50, cv=cv,
    scoring='accuracy',
    random_state=42, n_jobs=-1, verbose=1,
)
rand_search.fit(X_train, y_train)

print(f'Mejores parÃ¡metros: {rand_search.best_params_}')
print(f'Mejor CV accuracy : {rand_search.best_score_:.4f}')
best_adv_model = rand_search.best_estimator_


Modelo para tuning: HistGradientBoosting
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Mejores parÃ¡metros: {'min_samples_leaf': 20, 'max_leaf_nodes': 15, 'max_iter': 100, 'max_depth': 5, 'learning_rate': 0.1, 'l2_regularization': 0.0}
Mejor CV accuracy : 0.8149


## **6. Stacking Ensemble**

Combina los mejores modelos en un meta-modelo (Logistic Regression sobre las predicciones de los base models).


In [27]:
# Base estimators: los mejores de NB04 + el ganador del tuning de NB04.1
base_estimators = [
    ('hgb_tuned', best_adv_model),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('gb',  GradientBoostingClassifier(n_estimators=200, random_state=42)),
]

stacking = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1,
)

stack_cv = cross_val_score(stacking, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Stacking CV accuracy: {stack_cv.mean():.4f} +/- {stack_cv.std():.4f}')


Stacking CV accuracy: 0.8147 +/- 0.0058


## **7. EvaluaciÃ³n Comparativa en Validation Set**


In [28]:
# Entrenar todos los candidatos finales y evaluar en validation
candidates = {
    f'{best_adv_name} (tuned)': best_adv_model,
    'Stacking':                stacking,
}

eval_results = {}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    eval_results[name] = {
        'val_accuracy': accuracy_score(y_val, y_pred),
        'val_roc_auc':  roc_auc_score(y_val, y_proba),
    }
    print(f'{name:30s}  val_acc={accuracy_score(y_val, y_pred):.4f}  auc={roc_auc_score(y_val, y_proba):.4f}')


HistGradientBoosting (tuned)    val_acc=0.8115  auc=0.8990
Stacking                        val_acc=0.8092  auc=0.8994


In [29]:
eval_df = pd.DataFrame(eval_results).T.sort_values('val_accuracy', ascending=False)

fig = px.bar(
    eval_df.reset_index().rename(columns={'index': 'Modelo'}),
    x='Modelo', y='val_accuracy',
    title='Validation Accuracy â€” Comparativa Final',
    labels={'val_accuracy': 'Val Accuracy'},
    color='val_accuracy', color_continuous_scale='Greens',
)
if nb04_val_acc:
    fig.add_hline(y=nb04_val_acc, line_dash='dash', line_color='red',
                  annotation_text=f'NB04: {nb04_val_acc:.3f}')
fig.add_hline(y=TARGET_ACCURACY, line_dash='dot', line_color='orange',
              annotation_text=f'Target: {TARGET_ACCURACY}')
fig.update_layout(height=400, showlegend=False)
fig.show()


## **8. SelecciÃ³n y Guardado del Mejor Modelo**

Solo se guarda si supera el modelo de NB04 (comparaciÃ³n honesta).


In [30]:
best_name = eval_df.index[0]
best_val  = eval_df.loc[best_name, 'val_accuracy']
best_model_adv = candidates[best_name]

# Re-entrenar sobre todo el dataset de train para el modelo final
best_model_adv.fit(X_train, y_train)

os.makedirs('../../models', exist_ok=True)
joblib.dump(best_model_adv, MODEL_OUT_PATH)

metadata = {
    'experiment': 'NB04.1_Advanced_Models',
    'model': best_name,
    'val_accuracy': round(best_val, 4),
    'val_roc_auc':  round(eval_df.loc[best_name, 'val_roc_auc'], 4),
    'cv_accuracy':  round(rand_search.best_score_, 4),
    'target_accuracy': TARGET_ACCURACY,
    'target_achieved': bool(best_val >= TARGET_ACCURACY),
    'n_features': int(X.shape[1]),
    'n_train_samples': int(X_train.shape[0]),
}
with open(META_OUT_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Modelo guardado: {MODEL_OUT_PATH}')
print(f'Metadatos:       {META_OUT_PATH}')


Modelo guardado: ../../models/best_advanced_model.pkl
Metadatos:       ../../models/advanced_model_metadata.json


## **9. Resumen del Experimento**


In [31]:
print('=' * 60)
print('RESUMEN â€” EXPERIMENTO MODELOS AVANZADOS')
print('=' * 60)
print(f'  Modelo ganador  : {best_name}')
print(f'  Val Accuracy    : {best_val:.4f}')
print(f'  Val ROC-AUC     : {eval_df.loc[best_name, "val_roc_auc"]:.4f}')
if nb04_val_acc:
    delta = best_val - nb04_val_acc
    print(f'  NB04 referencia : {nb04_val_acc:.4f}')
    print(f'  Delta vs NB04   : {delta:+.4f}')
print(f'  Target ({TARGET_ACCURACY}): {"ALCANZADO" if best_val >= TARGET_ACCURACY else "NO alcanzado â€” continuar explorando"}')
print('=' * 60)
print()
if best_val >= TARGET_ACCURACY:
    print(f'  Usar {MODEL_OUT_PATH} en NB05 para predicciones.')
else:
    print('  Opciones para seguir mejorando:')
    print('    - Agregar features de interaccion (NB03.1)')
    print('    - Tuning con mÃ¡s iteraciones (n_iter > 50)')
    print('    - Neural Network (MLPClassifier)')


RESUMEN â€” EXPERIMENTO MODELOS AVANZADOS
  Modelo ganador  : HistGradientBoosting (tuned)
  Val Accuracy    : 0.8115
  Val ROC-AUC     : 0.8990
  NB04 referencia : 0.8080
  Delta vs NB04   : +0.0035
  Target (0.83): NO alcanzado â€” continuar explorando

  Opciones para seguir mejorando:
    - Agregar features de interaccion (NB03.1)
    - Tuning con mÃ¡s iteraciones (n_iter > 50)
    - Neural Network (MLPClassifier)
